# Домашнее задание: Занятие 34

**Тема: Ускорение обучения с GPU**

В этом задании вы закрепите работу с GPU, Mixed Precision, оптимизацией DataLoader и Gradient Accumulation. Каждое задание ссылается на конкретный слайд из презентации -- если забыли, вернитесь к нему.

---

## Часть 1: Теория (40 баллов)

Отвечайте своими словами, не копируйте.

### Вопрос 1 (8 баллов)

*(Слайды 5-6)*

а) Объясните разницу между CPU и GPU. Почему GPU ускоряет обучение нейросетей? Приведите аналогию.

б) У RTX 4070 около 5888 CUDA ядер, а у Intel i7 -- 8 ядер. Означает ли это, что GPU в 736 раз быстрее CPU? Почему да или нет?

**Ваш ответ:**

CPU — это универсальный процессор с небольшим количеством мощных ядер. Он хорошо подходит для последовательных задач и сложной логики.

GPU — это процессор с тысячами более простых ядер, которые могут выполнять множество одинаковых операций одновременно. Нейросети состоят в основном из операций с матрицами и тензорами, которые можно легко распараллелить, поэтому GPU может обрабатывать их намного быстрее.

Нет, это не означает, что GPU в 736 раз быстрее. CUDA-ядра и ядра CPU устроены по-разному и выполняют разные задачи. CPU-ядра намного мощнее и универсальнее, они выполняют сложные операции, ветвления и работу с памятью. CUDA-ядра проще и оптимизированы для параллельных математических операций.

### Вопрос 2 (8 баллов)

*(Слайды 7-8)*

а) Напишите 3 строки кода, которые: определяют устройство, создают модель Linear(100, 10) и переносят её на GPU.

б) Что произойдёт если модель на GPU, а входные данные на CPU? Напишите точное название ошибки и как её исправить.

**Ваш ответ:**

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = nn.Linear(100, 10)
model = model.to(device)

PyTorch выдаст ошибку RuntimeError, нужно перенести входные данные на то же устройство.

inputs = inputs.to(device)

### Вопрос 3 (8 баллов)

*(Слайды 11-13)*

а) Что делают параметры `pin_memory` и `num_workers` в DataLoader? Почему без них GPU может простаивать?

б) Что такое Mixed Precision Training? Почему называется "Mixed" (смешанная), а не просто float16? Какие две компоненты из torch.amp нужны и зачем каждая?

**Ваш ответ:**

num_workers задаёт количество процессов, которые загружают данные параллельно. Это ускоряет подготовку батчей.

pin_memory=True закрепляет память на CPU, чтобы данные быстрее копировались на GPU.

Если не использовать эти параметры, GPU может простаивать и ждать, пока CPU подготовит следующий батч данных.

Mixed Precision — это метод ускорения обучения, при котором часть вычислений выполняется в float16, а часть остаётся в float32.

Он называется "mixed" (смешанный), потому что используются два типа точности одновременно. Это нужно, чтобы сохранить стабильность обучения.

В torch.amp используются две основные компоненты:

- autocast, автоматически выполняет некоторые операции в float16 для ускорения и экономии памяти.

- GradScaler, масштабирует градиенты, чтобы избежать проблем с очень маленькими значениями (underflow), которые могут появляться при использовании float16.

### Вопрос 4 (8 баллов)

*(Слайд 14)*

а) Что такое Gradient Accumulation? В каких случаях он нужен?

б) У вас batch_size=32 и accumulation_steps=8. Какой эффективный batch size? Почему мы делим loss на accumulation_steps?

**Ваш ответ:**

Gradient Accumulation — это техника, при которой градиенты накапливаются за несколько батчей перед обновлением весов.

Она используется, когда GPU не хватает памяти для большого batch size. Мы делаем несколько маленьких батчей, суммируем градиенты и обновляем модель реже.

Мы делим loss на accumulation_steps, чтобы средний градиент был таким же, как если бы модель обучалась сразу на большом батче. Это делает обучение стабильным и эквивалентным использованию большого batch size.

32 × 8 = 256, эффективный batch size будет 256.

### Вопрос 5 (8 баллов)

*(Слайды 9, 16-17)*

Найдите 5 ошибок в коде ниже и исправьте их:

```python
model = CIFAR10_CNN()  # не перенесена на GPU
scaler = GradScaler()
losses = []

for epoch in range(10):
    for X_batch, y_batch in train_loader:
        # данные не перенесены на GPU
        optimizer.zero_grad()
        
        with autocast(device_type='cuda'):
            output = model(X_batch)
            loss = criterion(output, y_batch)
        
        loss.backward()            # не используется scaler
        optimizer.step()           # не используется scaler
        losses.append(loss)        # утечка памяти GPU
    
    # Оценка на тесте
    correct = 0
    total = 0
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        output = model(X_batch)    # нет eval() и no_grad()
        preds = output.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += len(y_batch)
```

Перепишите код правильно.

**Ваш ответ:**

model = CIFAR10_CNN().to(device)   # перенос модели на GPU
scaler = GradScaler()
losses = []

for epoch in range(10):
    model.train()
    
    for X_batch, y_batch in train_loader:
        # перенос данных на GPU
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        
        with autocast(device_type='cuda'):
            output = model(X_batch)
            loss = criterion(output, y_batch)
        
        # использование scaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        losses.append(loss.item())   # избегаем утечки памяти
    
    # Оценка на тесте
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            output = model(X_batch)
            preds = output.argmax(dim=1)

            correct += (preds == y_batch).sum().item()
            total += len(y_batch)

---

## Часть 2: Код (60 баллов)

### Подготовка

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torch.amp import autocast, GradScaler
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import time

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


In [ ]:
# Загружаем CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

train_data = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
test_data = datasets.CIFAR10('./data', train=False, download=True, transform=transform)

print(f'Train: {len(train_data)}')
print(f'Test:  {len(test_data)}')

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

100%|██████████| 170M/170M [00:02<00:00, 71.4MB/s]


Train: 50000
Test:  10000


### Задание 1: Проверка GPU и перенос данных (10 баллов)

*(Слайды 7-8)*

а) Выведите информацию о GPU: название, количество памяти, количество GPU.

б) Создайте тензор 1000x1000 на CPU, перенесите на GPU, убедитесь что он на GPU.

в) Создайте DataLoader для train и test с параметрами:
- batch_size = 128
- num_workers = 2, pin_memory = True
- shuffle = True для train, False для test

In [ ]:
# а) Информация о GPU
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Количество GPU:", torch.cuda.device_count())
    print("Память GPU (GB):", torch.cuda.get_device_properties(0).total_memory / 1024**3)
else:
    print("GPU нет")

GPU нет


In [ ]:
# б) Перенос тензора
tensor_cpu = torch.randn(1000, 1000)

tensor_gpu = tensor_cpu.to(device)

print("Устройство тензора:", tensor_gpu.device)


Устройство тензора: cpu


In [ ]:
# в) DataLoader
BATCH_SIZE = 128

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f'Батчей в train: {len(train_loader)}')
print(f'Батчей в test:  {len(test_loader)}')

for images, labels in train_loader:
    print(f'Images: {images.shape}')
    print(f'Labels: {labels.shape}')
    break

Батчей в train: 391
Батчей в test:  79


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Images: torch.Size([128, 3, 32, 32])
Labels: torch.Size([128])


### Задание 2: Постройте CNN для CIFAR-10 (10 баллов)

*(Слайд 8)*

Создайте CNN:
- Conv2d(3, 32, 3, padding=1) + BatchNorm2d + ReLU + MaxPool2d(2)
- Conv2d(32, 64, 3, padding=1) + BatchNorm2d + ReLU + MaxPool2d(2)
- Conv2d(64, 128, 3, padding=1) + BatchNorm2d + ReLU + MaxPool2d(2)
- Flatten + Dropout(0.5) + Linear(128*4*4, 256) + ReLU + Linear(256, 10)

Перенесите на GPU. Выведите количество параметров.

In [ ]:
class CIFAR10_CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128*4*4, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x

model = CIFAR10_CNN().to(device)
print(model)
print(f'\nПараметров: {sum(p.numel() for p in model.parameters()):,}')

# Проверка: пропустите один батч
with torch.no_grad():
    test_input = torch.randn(2, 3, 32, 32).to(device)
    test_output = model(test_input)
    print(f'Вход: {test_input.shape} -> Выход: {test_output.shape}')

CIFAR10_CNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.5, inplace=False)
    (2): Linear(in_features=2048, out_features=256, b

### Задание 3: Training Loop с GPU и Mixed Precision (15 баллов)

*(Слайды 8, 12-13)*

Напишите полный цикл обучения на 15 эпох с Mixed Precision:
- CrossEntropyLoss, Adam с lr=0.001
- Используйте autocast и GradScaler
- Сохраняйте train_loss и test_accuracy на каждой эпохе
- Не забудьте: model.train(), model.eval(), .to(device), torch.no_grad()
- Замерьте общее время обучения
- Выводите прогресс каждые 3 эпохи

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scaler = GradScaler()

NUM_EPOCHS = 15
train_losses = []
test_accs = []

start_time = time.time()

for epoch in range(NUM_EPOCHS):

    # ---- TRAIN ----
    model.train()

    running_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        with autocast(device_type=device.type):

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)


    # ---- TEST ----

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            preds = outputs.argmax(dim=1)

            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    acc = correct / total
    test_accs.append(acc)

    if (epoch + 1) % 3 == 0:
        print(f'Epoch {epoch+1}/{NUM_EPOCHS}  '
              f'Loss: {train_losses[-1]:.4f}  '
              f'Acc: {test_accs[-1]:.2%}')

total_time = time.time() - start_time
print(f'\nОбучение заняло: {total_time:.1f} сек ({total_time/NUM_EPOCHS:.1f} сек/эпоха)')
print(f'Лучшая accuracy: {max(test_accs):.2%}')

/tmp/ipykernel_471/4285473272.py:3: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = GradScaler()


### Задание 4: Графики обучения (5 баллов)

Постройте два графика рядом: Train Loss и Test Accuracy по эпохам.

In [ ]:
# Ваш код здесь

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(train_losses)
plt.title("Train Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.subplot(1,2,2)
plt.plot(test_accs)
plt.title("Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.show()

### Задание 5: Эксперимент -- float32 vs Mixed Precision (15 баллов)

*(Слайды 12-13)*

Обучите ту же модель CIFAR10_CNN двумя способами:
1. Обычное обучение (float32)
2. Mixed Precision (autocast + GradScaler)

Для каждого:
- Обучите на 10 эпох (lr=0.001, batch_size=128)
- Засеките время обучения
- Замерьте пиковую память GPU (torch.cuda.max_memory_allocated)
- Запишите финальную accuracy

Постройте:
1. График Test Accuracy по эпохам (оба на одном графике)
2. Барчарт: время обучения и пиковая память

Напишите вывод: стоит ли использовать Mixed Precision?

In [ ]:
# Ваш код здесь
def train_model(use_amp, num_epochs=10):

    model = CIFAR10_CNN().to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    scaler = GradScaler()

    losses = []
    accs = []

    start = time.time()

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    for epoch in range(num_epochs):

        model.train()
        running_loss = 0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            if use_amp:

                with autocast(device_type=device.type):

                    outputs = model(X_batch)
                    loss = criterion(outputs, y_batch)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            else:

                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)

                loss.backward()
                optimizer.step()

            running_loss += loss.item()

        losses.append(running_loss/len(train_loader))


        # TEST
        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for X_batch, y_batch in test_loader:

                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                outputs = model(X_batch)

                preds = outputs.argmax(dim=1)

                correct += (preds == y_batch).sum().item()
                total += y_batch.size(0)

        accs.append(correct/total)

    total_time = time.time() - start

    if torch.cuda.is_available():
        peak_memory = torch.cuda.max_memory_allocated()/1024**2
    else:
        peak_memory = 0

    return losses, accs, total_time, peak_memory


In [ ]:
# Графики сравнения

plt.plot(acc_fp32, label="float32")
plt.plot(acc_amp, label="AMP")

plt.title("Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend()

plt.show()

**Ваш вывод:**



### Задание 6: Визуализация ошибок модели (5 баллов)

Возьмите лучшую модель из Задания 5. Найдите 16 картинок, на которых модель ОШИБЛАСЬ. Покажите их с предсказанием и правильным классом.

In [ ]:
# Ваш код здесь

model.eval()

wrong_images = []
wrong_preds = []
wrong_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        preds = outputs.argmax(dim=1)

        wrong_mask = preds != labels

        if wrong_mask.any():

            wrong_images.extend(images[wrong_mask].cpu())
            wrong_preds.extend(preds[wrong_mask].cpu())
            wrong_labels.extend(labels[wrong_mask].cpu())

        if len(wrong_images) >= 16:
            break


# Показываем 16 ошибок

plt.figure(figsize=(10,10))

for i in range(16):

    plt.subplot(4,4,i+1)

    img = wrong_images[i].permute(1,2,0).numpy()

    # убираем нормализацию
    img = img * np.array((0.2470,0.2435,0.2616)) + np.array((0.4914,0.4822,0.4465))
    img = np.clip(img,0,1)

    plt.imshow(img)

    pred = class_names[wrong_preds[i]]
    true = class_names[wrong_labels[i]]

    plt.title(f'Pred: {pred}\nTrue: {true}', fontsize=9)

    plt.axis("off")

plt.tight_layout()
plt.show()

---

## Часть 3: Бонус (20 баллов)

### Бонус 1: Gradient Accumulation эксперимент (10 баллов)

*(Слайд 14)*

Обучите модель с тремя конфигурациями:
1. batch=32, accum=1 (эффективный batch=32)
2. batch=32, accum=4 (эффективный batch=128)
3. batch=128, accum=1 (реальный batch=128)

Конфигурации 2 и 3 должны дать похожие результаты (одинаковый эффективный batch). Проверьте это.

Постройте график accuracy и напишите вывод.

In [ ]:
# Ваш код здесь



**Ваш вывод:**



### Бонус 2: Чеклист оптимизации на практике (10 баллов)

*(Слайд 21)*

Примените чеклист оптимизации пошагово. Обучите модель 5 эпох каждый раз, добавляя по одной оптимизации:

1. Базовый: CPU, num_workers=0, pin_memory=False, float32
2. +GPU: то же, но model и data на GPU
3. +DataLoader: num_workers=2, pin_memory=True
4. +AMP: Mixed Precision

Замерьте время каждой конфигурации. Постройте барчарт времени обучения. Какой шаг дал наибольшее ускорение?

In [ ]:
# Ваш код здесь



**Ваш вывод:**

